In [3]:
!pip install -q datasets transformers accelerate scikit-learn

In [4]:
import torch
import transformers
import datasets

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("Datasets:", datasets.__version__)
print("GPU available:", torch.cuda.is_available())

PyTorch: 2.11.0+cu128
Transformers: 5.13.1
Datasets: 4.0.0
GPU available: True


In [5]:
from datasets import load_dataset

dataset = load_dataset("stanfordnlp/imdb")

print(dataset)

README.md:   0%|          | 0.00/7.81k [00:00<?, ?B/s]

plain_text/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 21.0MB            

plain_text/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

plain_text/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 20.5MB            

plain_text/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

...

plain_text/unsupervised-00000-of-00001.p(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label'],
        num_rows: 50000
    })
})


In [6]:
small_train = dataset["train"].shuffle(seed=42).select(range(2000))
small_test = dataset["test"].shuffle(seed=42).select(range(500))

print("Training samples:", len(small_train))
print("Testing samples:", len(small_test))

Training samples: 2000
Testing samples: 500


In [7]:
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    pipeline
)
import numpy as np
from sklearn.metrics import accuracy_score

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(
    "distilbert-base-uncased"
)

# Tokenization function
def tokenize(batch):
    return tokenizer(
        batch["text"],
        padding="max_length",
        truncation=True,
        max_length=128
    )

# Tokenize datasets
train_ds = small_train.map(
    tokenize,
    batched=True
)

test_ds = small_test.map(
    tokenize,
    batched=True
)

print("Tokenization completed!")

# Load DistilBERT
model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=2
)

print("Model loaded!")

# Training arguments
args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=2,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    eval_strategy="epoch",
    logging_steps=50,
    report_to="none"
)

# Accuracy
def compute_metrics(pred):
    predictions = np.argmax(
        pred.predictions,
        axis=1
    )

    return {
        "accuracy": accuracy_score(
            pred.label_ids,
            predictions
        )
    }

# Trainer
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=test_ds,
    compute_metrics=compute_metrics
)

# Train
print("Starting training...")
trainer.train()

# Evaluate
metrics = trainer.evaluate()

print("\nEvaluation Metrics:")
print(metrics)

# Save model
model.save_pretrained(
    "./fine_tuned_distilbert_imdb"
)

tokenizer.save_pretrained(
    "./fine_tuned_distilbert_imdb"
)

print("\nModel saved successfully!")

# Test the model
classifier = pipeline(
    "text-classification",
    model="./fine_tuned_distilbert_imdb",
    tokenizer="./fine_tuned_distilbert_imdb"
)

review = "This movie was amazing and I really enjoyed it!"

result = classifier(review)

print("\nTest Review:")
print(review)

print("\nPrediction:")
print(result)

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Tokenization completed!


model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

...


Evaluation Metrics:
{'eval_loss': 0.6214647889137268, 'eval_accuracy': 0.812}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Model saved successfully!


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]


Test Review:
This movie was amazing and I really enjoyed it!

Prediction:
[{'label': 'LABEL_1', 'score': 0.9897118806838989}]
